In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [7]:
questions = [
    "hi", "hello", "how are you", "what is your name",
    "what can you do", "tell me about ai",
    "i am sad", "i am happy", "good morning", "good night",
    "tell me a joke", "bye"
]

answers = [
    "hi there how are you",
    "hello nice to meet you",
    "i am fine how can i help you today",
    "i am a chatbot i can assist you",
    "i can chat with you answer questions and help you learn",
    "artificial intelligence is the simulation of human intelligence by machines",
    "i am sorry to hear that don't worry things will get better",
    "that's great keep smiling and stay positive",
    "good morning have a nice day ahead",
    "good night sleep well and take care",
    "why did the computer go to school because it wanted to improve its skills haha",
    "goodbye see you later take care"
]

In [19]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(questions + answers)

X = tokenizer.texts_to_sequences(questions)
Y = tokenizer.texts_to_sequences(answers)

max_len = max(max(len(x) for x in X), max(len(y) for y in Y))

X = pad_sequences(X, maxlen=max_len, padding='post')
Y = pad_sequences(Y, maxlen=max_len, padding='post')

vocab_size = len(tokenizer.word_index) + 1

In [20]:
reverse_word_index = {v:k for k,v in tokenizer.word_index.items()}

In [21]:
Y = np.expand_dims(Y, axis=-1)

In [22]:
model = Sequential([
    Embedding(vocab_size, 128),
    LSTM(128, return_sequences=True),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
model.fit(
    X, Y,
    epochs=300,
    verbose=0
)

In [11]:
reverse_word_index = {v:k for k,v in tokenizer.word_index.items()}

def chatbot(text):

    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)[0]

    words = []

    for i in range(len(pred)):
        idx = np.argmax(pred[i])

        if idx != 0:
            word = reverse_word_index.get(idx, "")
            if word:
                words.append(word)

    return " ".join(words)

In [24]:
print("Input: hello")
print("Output:", chatbot("hello"))

print("\nInput: what is ai")
print("Output:", chatbot("what is ai"))

print("\nInput: i am sad")
print("Output:", chatbot("i am sad"))

Input: hello
Output: hello nice to meet you

Input: what is ai
Output: i am a can can you you

Input: i am sad
Output: i am sorry to hear that don't worry things will get better


In [9]:
reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}

def chatbot_response(text):

    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)[0]

    words = []

    for i in range(len(pred)):
        idx = np.argmax(pred[i])

        if idx != 0:
            word = reverse_word_index.get(idx, "")
            if word:
                words.append(word)

    return " ".join(words)

In [25]:
print("🤖 Chatbot is ready! Type 'exit' to stop.\n")

while True:

    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye! 👋")
        break

    response = chatbot_response(user)

    print("Bot:", response)

🤖 Chatbot is ready! Type 'exit' to stop.

You: hi
Bot: hi there how are you
You: how are you
Bot: i am fine how can i help you today
You: what is your name
Bot: i am a chatbot i can assist you
You: what can you do
Bot: i can chat with you answer questions and help you learn
You: tell me about ai
Bot: why intelligence is the simulation of human intelligence by machines
You: good night 
Bot: good night sleep well and take care
You: bye
Bot: goodbye see you later take care
You: exit
Bot: Goodbye! 👋
